# ☁️ Notebook 05 - Deploy AWS Lambda

## Tech Challenge 4 - LSTM VALE3

---

### Objetivos deste Notebook:

1. **Configurar alerta de billing** (segurança contra gastos)
2. **Criar imagem Docker** otimizada para Lambda
3. **Subir para ECR** (Elastic Container Registry)
4. **Criar função Lambda**
5. **Configurar API Gateway** (URL pública)
6. **Testar a API em produção**
7. **Checklist de limpeza** (pós-vídeo)

---

### Arquitetura do Deploy

```
Usuário/Cliente
      │
      ▼
┌─────────────────┐
│  API Gateway    │  ← URL pública (https://xxx.amazonaws.com)
│  (HTTP API)     │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  AWS Lambda     │  ← Executa o código (paga por execução)
│  (Container)    │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  ECR            │  ← Armazena a imagem Docker
│  (Imagem Docker)│
└─────────────────┘
```

---

### ⚠️ IMPORTANTE: Custos Estimados

| Serviço | Custo Estimado |
|---------|----------------|
| ECR (500MB) | ~$0.05/mês |
| Lambda (100 execuções) | ~$0.00 |
| API Gateway (100 requests) | ~$0.00 |
| **Total** | **< $0.50** |

**Lembre-se de deletar tudo após gravar o vídeo!**

## 1. Pré-Requisitos

### Verificar instalações necessárias

In [1]:
# =============================================================================
# VERIFICAR PRÉ-REQUISITOS
# =============================================================================

import subprocess
import shutil

print("VERIFICAÇÃO DE PRÉ-REQUISITOS")
print("=" * 50)

# Verifica AWS CLI
aws_ok = shutil.which('aws') is not None
print(f"\n{'✓' if aws_ok else '✗'} AWS CLI: {'Instalado' if aws_ok else 'NÃO INSTALADO'}")
if not aws_ok:
    print("   Instale: https://docs.aws.amazon.com/cli/latest/userguide/getting-started-install.html")

# Verifica Docker
docker_ok = shutil.which('docker') is not None
print(f"{'✓' if docker_ok else '✗'} Docker: {'Instalado' if docker_ok else 'NÃO INSTALADO'}")
if not docker_ok:
    print("   Instale: https://docs.docker.com/get-docker/")

# Verifica configuração AWS
if aws_ok:
    try:
        result = subprocess.run(['aws', 'sts', 'get-caller-identity'], 
                                capture_output=True, text=True, timeout=10)
        if result.returncode == 0:
            print(f"✓ AWS CLI: Configurado e autenticado")
            import json
            identity = json.loads(result.stdout)
            print(f"   Account ID: {identity['Account']}")
        else:
            print(f"✗ AWS CLI: Não configurado")
            print(f"   Execute: aws configure")
    except:
        print(f"✗ AWS CLI: Erro ao verificar configuração")

print("\n" + "=" * 50)
if aws_ok and docker_ok:
    print("✓ Todos os pré-requisitos atendidos!")
else:
    print("⚠️ Instale os itens faltantes antes de continuar.")

VERIFICAÇÃO DE PRÉ-REQUISITOS

✓ AWS CLI: Instalado
✓ Docker: Instalado
✓ AWS CLI: Configurado e autenticado
   Account ID: 299579972912

✓ Todos os pré-requisitos atendidos!


## 2. 🚨 Configurar Alerta de Billing (FAÇA PRIMEIRO!)

Antes de qualquer coisa, vamos configurar um alerta para evitar surpresas na fatura.

### Passo a Passo no Console AWS:

1. Acesse: https://console.aws.amazon.com/billing/home#/budgets
2. Clique em **"Create budget"**
3. Selecione **"Cost budget"** → Next
4. Configure:
   - Budget name: `tech-challenge-alert`
   - Period: `Monthly`
   - Budget amount: `$5.00`
5. Em **Alerts**:
   - Threshold: `50%` ($2.50)
   - Email: seu email
6. Adicione outro alerta em `80%` ($4.00)
7. **Create budget**

---

**Confirme que criou o alerta antes de prosseguir!**

In [2]:
# =============================================================================
# CONFIRMAÇÃO DO ALERTA DE BILLING
# =============================================================================

confirmacao = input("Você configurou o alerta de billing? (sim/nao): ").strip().lower()

if confirmacao in ['sim', 's', 'yes', 'y']:
    print("\n✓ Ótimo! Podemos prosseguir com segurança.")
else:
    print("\n⚠️ PARE AQUI!")
    print("Configure o alerta antes de continuar.")
    print("Acesse: https://console.aws.amazon.com/billing/home#/budgets")

Você configurou o alerta de billing? (sim/nao):  sim



✓ Ótimo! Podemos prosseguir com segurança.


## 3. Configurar Variáveis do Projeto

In [3]:
# =============================================================================
# CONFIGURAÇÕES DO DEPLOY
# =============================================================================

import subprocess
import json

# Obtém Account ID automaticamente
result = subprocess.run(['aws', 'sts', 'get-caller-identity'], capture_output=True, text=True)
identity = json.loads(result.stdout)
AWS_ACCOUNT_ID = identity['Account']

# Configurações
AWS_REGION = "sa-east-1"  # Região mais barata geralmente
ECR_REPO_NAME = "lstm-vale3-api"
LAMBDA_FUNCTION_NAME = "lstm-vale3-predict"
LAMBDA_TIMEOUT = 60  # segundos (modelo pode demorar para carregar)
LAMBDA_MEMORY = 1024  # MB (TensorFlow precisa de memória)

# URI completa do ECR
ECR_URI = f"{AWS_ACCOUNT_ID}.dkr.ecr.{AWS_REGION}.amazonaws.com/{ECR_REPO_NAME}"

print("CONFIGURAÇÕES DO DEPLOY")
print("=" * 50)
print(f"\nAWS Account ID: {AWS_ACCOUNT_ID}")
print(f"Região: {AWS_REGION}")
print(f"Repositório ECR: {ECR_REPO_NAME}")
print(f"Função Lambda: {LAMBDA_FUNCTION_NAME}")
print(f"Timeout: {LAMBDA_TIMEOUT}s")
print(f"Memória: {LAMBDA_MEMORY}MB")
print(f"\nECR URI: {ECR_URI}")

CONFIGURAÇÕES DO DEPLOY

AWS Account ID: 299579972912
Região: sa-east-1
Repositório ECR: lstm-vale3-api
Função Lambda: lstm-vale3-predict
Timeout: 60s
Memória: 1024MB

ECR URI: 299579972912.dkr.ecr.sa-east-1.amazonaws.com/lstm-vale3-api


In [4]:
# =============================================================================
# SALVAR CONFIGURAÇÕES PARA USO NOS COMANDOS
# =============================================================================

# Salva em arquivo para referência
config_deploy = {
    'aws_account_id': AWS_ACCOUNT_ID,
    'aws_region': AWS_REGION,
    'ecr_repo_name': ECR_REPO_NAME,
    'ecr_uri': ECR_URI,
    'lambda_function_name': LAMBDA_FUNCTION_NAME,
    'lambda_timeout': LAMBDA_TIMEOUT,
    'lambda_memory': LAMBDA_MEMORY
}

with open('../config_deploy.json', 'w') as f:
    json.dump(config_deploy, f, indent=2)

print("✓ Configurações salvas em config_deploy.json")

✓ Configurações salvas em config_deploy.json


## 4. Criar Dockerfile Otimizado

O TensorFlow é grande (~500MB), então vamos otimizar a imagem.

In [5]:
%%writefile ../Dockerfile
# =============================================================================
# Dockerfile para AWS Lambda com TensorFlow
# Otimizado para menor tamanho possível
# =============================================================================

FROM public.ecr.aws/lambda/python:3.11

# Variáveis de ambiente para otimização
ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1
ENV TF_CPP_MIN_LOG_LEVEL=2

# Copia e instala dependências primeiro (cache de camadas)
COPY requirements-deploy.txt ${LAMBDA_TASK_ROOT}/
RUN pip install --no-cache-dir -r requirements-deploy.txt

# Copia código da API
COPY src/api/ ${LAMBDA_TASK_ROOT}/src/api/
COPY src/__init__.py ${LAMBDA_TASK_ROOT}/src/

# Copia modelo e arquivos necessários
COPY models/lstm_vale3.keras ${LAMBDA_TASK_ROOT}/models/
COPY models/scaler.joblib ${LAMBDA_TASK_ROOT}/models/
COPY models/metricas.json ${LAMBDA_TASK_ROOT}/models/

# Copia configuração
RUN mkdir -p ${LAMBDA_TASK_ROOT}/data/processed
COPY data/processed/config.json ${LAMBDA_TASK_ROOT}/data/processed/

# Handler do Lambda
CMD ["src.api.main.handler"]

Overwriting ../Dockerfile


In [6]:
%%writefile ../requirements-deploy.txt
# =============================================================================
# Requirements para Deploy AWS Lambda
# Versões fixas para estabilidade
# =============================================================================

# API Framework
fastapi==0.109.0
mangum==0.17.0
pydantic==2.5.3

# ML - TensorFlow CPU only (menor tamanho)
tensorflow-cpu==2.15.0
numpy==1.26.3
scikit-learn==1.4.0
joblib==1.3.2

# Dados
pandas==2.1.4
yfinance==0.2.36

Overwriting ../requirements-deploy.txt


In [7]:
print("✓ Dockerfile e requirements-deploy.txt criados!")

✓ Dockerfile e requirements-deploy.txt criados!


## 5. Criar Repositório ECR

ECR (Elastic Container Registry) é onde vamos armazenar a imagem Docker.

In [8]:
# =============================================================================
# CRIAR REPOSITÓRIO ECR
# =============================================================================

import subprocess

print("CRIANDO REPOSITÓRIO ECR")
print("=" * 50)

# Comando para criar repositório
cmd = f"aws ecr create-repository --repository-name {ECR_REPO_NAME} --region {AWS_REGION}"

print(f"\nExecutando: {cmd}\n")

result = subprocess.run(cmd.split(), capture_output=True, text=True)

if result.returncode == 0:
    print("✓ Repositório criado com sucesso!")
    repo_info = json.loads(result.stdout)
    print(f"   URI: {repo_info['repository']['repositoryUri']}")
elif "RepositoryAlreadyExistsException" in result.stderr:
    print("✓ Repositório já existe (ok, vamos usar o existente)")
else:
    print(f"✗ Erro: {result.stderr}")

CRIANDO REPOSITÓRIO ECR

Executando: aws ecr create-repository --repository-name lstm-vale3-api --region sa-east-1

✓ Repositório já existe (ok, vamos usar o existente)


## 6. Build e Push da Imagem Docker

⏱️ **Este passo pode demorar 5-10 minutos** (TensorFlow é grande)

In [9]:
# =============================================================================
# LOGIN NO ECR
# =============================================================================

print("LOGIN NO ECR")
print("=" * 50)

# Comando de login
login_cmd = f"aws ecr get-login-password --region {AWS_REGION}"
docker_login_cmd = f"docker login --username AWS --password-stdin {AWS_ACCOUNT_ID}.dkr.ecr.{AWS_REGION}.amazonaws.com"

print("Executando login...")

# Executa login em duas etapas (pipe)
p1 = subprocess.Popen(login_cmd.split(), stdout=subprocess.PIPE)
p2 = subprocess.Popen(docker_login_cmd.split(), stdin=p1.stdout, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
p1.stdout.close()
output, error = p2.communicate()

if p2.returncode == 0:
    print("✓ Login realizado com sucesso!")
else:
    print(f"✗ Erro no login: {error.decode()}")

LOGIN NO ECR
Executando login...
✓ Login realizado com sucesso!


In [10]:
# =============================================================================
# BUILD DA IMAGEM DOCKER
# =============================================================================

import os

print("BUILD DA IMAGEM DOCKER")
print("=" * 50)
print("\n⏱️  Isso pode demorar 5-10 minutos...\n")

# Muda para diretório raiz do projeto
os.chdir('..')
print(f"Diretório atual: {os.getcwd()}")

# Comando de build
build_cmd = f"docker build --platform linux/amd64 -t {ECR_REPO_NAME}:latest ."
print(f"\nExecutando: {build_cmd}\n")
print("-" * 50)

# Executa build (mostrando output)
result = subprocess.run(build_cmd.split(), capture_output=False)

if result.returncode == 0:
    print("-" * 50)
    print("\n✓ Build concluído com sucesso!")
else:
    print("\n✗ Erro no build. Verifique os logs acima.")

# Volta para pasta notebooks
os.chdir('notebooks')

BUILD DA IMAGEM DOCKER

⏱️  Isso pode demorar 5-10 minutos...

Diretório atual: /Users/mariaaraujo/Documents/tech-challenge-4

Executando: docker build --platform linux/amd64 -t lstm-vale3-api:latest .

--------------------------------------------------


#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 1.16kB done
#1 DONE 0.0s

#2 [internal] load metadata for public.ecr.aws/lambda/python:3.11
#2 DONE 1.6s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [internal] load build context
#4 transferring context: 1.60kB done
#4 DONE 0.0s

#5 [ 1/10] FROM public.ecr.aws/lambda/python:3.11@sha256:f62db4e2adf9e2445f112dca3650f2a7370f7ab765c7e98db9214d8d69fae9f1
#5 resolve public.ecr.aws/lambda/python:3.11@sha256:f62db4e2adf9e2445f112dca3650f2a7370f7ab765c7e98db9214d8d69fae9f1 0.0s done
#5 CACHED

#6 [ 2/10] COPY requirements-deploy.txt /var/task/
#6 DONE 0.0s

#7 [ 3/10] RUN pip install --no-cache-dir -r requirements-deploy.txt
#7 1.204 Collecting fastapi==0.109.0 (from -r requirements-deploy.txt (line 7))
#7 2.344   Downloading fastapi-0.109.0-py3-none-any.whl.metadata (24 kB)
#7 2.410 Collecting mangum==0.17.0 (from


✗ Erro no build. Verifique os logs acima.


#7 ERROR: process "/bin/sh -c pip install --no-cache-dir -r requirements-deploy.txt" did not complete successfully: exit code: 1
------
 > [ 3/10] RUN pip install --no-cache-dir -r requirements-deploy.txt:
16.73 error: subprocess-exited-with-error
16.73 
16.73 × pip subprocess to install build dependencies did not run successfully.
16.73 │ exit code: 1
16.73 ╰─> See above for output.
16.73 
16.73 note: This error originates from a subprocess, and is likely not a problem with pip.
16.90 
16.90 [notice] A new release of pip is available: 24.0 -> 25.3
16.90 [notice] To update, run: pip install --upgrade pip
------
Dockerfile:15
--------------------
  13 |     # Copia e instala dependências primeiro (cache de camadas)
  14 |     COPY requirements-deploy.txt ${LAMBDA_TASK_ROOT}/
  15 | >>> RUN pip install --no-cache-dir -r requirements-deploy.txt
  16 |     
  17 |     # Copia código da API
--------------------
ERROR: failed to solve: process "/bin/sh -c pip install --no-cache-dir -r requir

In [11]:
# =============================================================================
# TAG E PUSH PARA ECR
# =============================================================================

print("TAG E PUSH PARA ECR")
print("=" * 50)

# Tag da imagem
tag_cmd = f"docker tag {ECR_REPO_NAME}:latest {ECR_URI}:latest"
print(f"\n1. Tag: {tag_cmd}")
result = subprocess.run(tag_cmd.split(), capture_output=True, text=True)
if result.returncode == 0:
    print("   ✓ Tag aplicada")
else:
    print(f"   ✗ Erro: {result.stderr}")

# Push para ECR
push_cmd = f"docker push {ECR_URI}:latest"
print(f"\n2. Push: {push_cmd}")
print("   ⏱️  Enviando imagem (pode demorar alguns minutos)...\n")

result = subprocess.run(push_cmd.split(), capture_output=False)

if result.returncode == 0:
    print("\n✓ Push concluído com sucesso!")
    print(f"   Imagem disponível em: {ECR_URI}:latest")
else:
    print("\n✗ Erro no push. Verifique os logs acima.")

TAG E PUSH PARA ECR

1. Tag: docker tag lstm-vale3-api:latest 299579972912.dkr.ecr.sa-east-1.amazonaws.com/lstm-vale3-api:latest
   ✓ Tag aplicada

2. Push: docker push 299579972912.dkr.ecr.sa-east-1.amazonaws.com/lstm-vale3-api:latest
   ⏱️  Enviando imagem (pode demorar alguns minutos)...

The push refers to repository [299579972912.dkr.ecr.sa-east-1.amazonaws.com/lstm-vale3-api]
529eaa13a224: Waiting
d6f953ce0616: Waiting
b1bbbcf4f28f: Waiting
370ec26734e0: Waiting
8215b8dd3995: Waiting
28d559187d93: Waiting
4b3c413086c4: Waiting
d3c89ac016dc: Waiting
f004b1bfc6ea: Waiting
43b1ad1e1e0e: Waiting
bb432596732e: Waiting
5448dd0d9f22: Waiting
f128a7ec450a: Waiting
32fc9619ff98: Waiting
485553fe8b6b: Waiting
e996daec7f1b: Waiting
1079c59499ac: Waiting
dcb6e7fdabee: Waiting
11616e4cdd01: Waiting
8727d932a40b: Waiting
c1847bb8e538: Waiting
665bf2503e0f: Waiting
7738dd62ffc8: Waiting
f515e2b6d4ed: Waiting
1552a1ed02fc: Waiting
96ebc51df618: Waiting
7738dd62ffc8: Waiting
f515e2b6d4ed: Waiting

KeyboardInterrupt: 

## 7. Criar Role IAM para Lambda

O Lambda precisa de permissões para executar.

In [ ]:
# =============================================================================
# CRIAR ROLE IAM PARA LAMBDA
# =============================================================================

print("CRIAR ROLE IAM")
print("=" * 50)

ROLE_NAME = "lambda-lstm-vale3-role"

# Trust policy (permite Lambda assumir a role)
trust_policy = '''{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Principal": {
        "Service": "lambda.amazonaws.com"
      },
      "Action": "sts:AssumeRole"
    }
  ]
}'''

# Salva policy em arquivo temporário
with open('/tmp/trust-policy.json', 'w') as f:
    f.write(trust_policy)

# Cria a role
create_role_cmd = f"aws iam create-role --role-name {ROLE_NAME} --assume-role-policy-document file:///tmp/trust-policy.json"

result = subprocess.run(create_role_cmd.split(), capture_output=True, text=True)

if result.returncode == 0:
    print("✓ Role criada com sucesso!")
    role_info = json.loads(result.stdout)
    ROLE_ARN = role_info['Role']['Arn']
    print(f"   ARN: {ROLE_ARN}")
elif "EntityAlreadyExists" in result.stderr:
    print("✓ Role já existe")
    # Obtém ARN da role existente
    get_role_cmd = f"aws iam get-role --role-name {ROLE_NAME}"
    result = subprocess.run(get_role_cmd.split(), capture_output=True, text=True)
    role_info = json.loads(result.stdout)
    ROLE_ARN = role_info['Role']['Arn']
    print(f"   ARN: {ROLE_ARN}")
else:
    print(f"✗ Erro: {result.stderr}")
    ROLE_ARN = None

In [ ]:
# =============================================================================
# ANEXAR POLÍTICA DE EXECUÇÃO BÁSICA
# =============================================================================

print("\nANEXAR POLÍTICA DE EXECUÇÃO")
print("=" * 50)

# Política básica de execução Lambda
attach_cmd = f"aws iam attach-role-policy --role-name {ROLE_NAME} --policy-arn arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"

result = subprocess.run(attach_cmd.split(), capture_output=True, text=True)

if result.returncode == 0 or "already attached" in result.stderr.lower():
    print("✓ Política anexada com sucesso!")
else:
    print(f"✗ Erro: {result.stderr}")

# Aguarda propagação da role
print("\n⏱️  Aguardando propagação da role (10 segundos)...")
import time
time.sleep(10)
print("✓ Pronto!")

## 8. Criar Função Lambda

In [ ]:
# =============================================================================
# CRIAR FUNÇÃO LAMBDA
# =============================================================================

print("CRIAR FUNÇÃO LAMBDA")
print("=" * 50)

create_lambda_cmd = f"""aws lambda create-function \
    --function-name {LAMBDA_FUNCTION_NAME} \
    --package-type Image \
    --code ImageUri={ECR_URI}:latest \
    --role {ROLE_ARN} \
    --timeout {LAMBDA_TIMEOUT} \
    --memory-size {LAMBDA_MEMORY} \
    --region {AWS_REGION}"""

print(f"Criando função Lambda...\n")

# Executa (tratando quebras de linha)
result = subprocess.run(create_lambda_cmd.replace('\\\n', ' ').split(), capture_output=True, text=True)

if result.returncode == 0:
    print("✓ Função Lambda criada com sucesso!")
    lambda_info = json.loads(result.stdout)
    LAMBDA_ARN = lambda_info['FunctionArn']
    print(f"   ARN: {LAMBDA_ARN}")
elif "ResourceConflictException" in result.stderr:
    print("✓ Função já existe. Atualizando...")
    
    # Atualiza código da função existente
    update_cmd = f"aws lambda update-function-code --function-name {LAMBDA_FUNCTION_NAME} --image-uri {ECR_URI}:latest --region {AWS_REGION}"
    result = subprocess.run(update_cmd.split(), capture_output=True, text=True)
    
    if result.returncode == 0:
        print("✓ Função atualizada com sucesso!")
        lambda_info = json.loads(result.stdout)
        LAMBDA_ARN = lambda_info['FunctionArn']
    else:
        print(f"✗ Erro na atualização: {result.stderr}")
else:
    print(f"✗ Erro: {result.stderr}")

In [ ]:
# =============================================================================
# AGUARDAR FUNÇÃO FICAR ATIVA
# =============================================================================

import time

print("\nAGUARDANDO FUNÇÃO FICAR ATIVA")
print("=" * 50)

for i in range(30):  # Tenta por até 5 minutos
    get_function_cmd = f"aws lambda get-function --function-name {LAMBDA_FUNCTION_NAME} --region {AWS_REGION}"
    result = subprocess.run(get_function_cmd.split(), capture_output=True, text=True)
    
    if result.returncode == 0:
        func_info = json.loads(result.stdout)
        state = func_info['Configuration']['State']
        print(f"   Status: {state}", end='\r')
        
        if state == 'Active':
            print(f"\n\n✓ Função está ATIVA!")
            break
    
    time.sleep(10)
else:
    print("\n⚠️ Timeout aguardando função. Verifique no console AWS.")

## 9. Criar API Gateway

API Gateway fornece a URL pública para acessar o Lambda.

In [ ]:
# =============================================================================
# CRIAR API GATEWAY (HTTP API)
# =============================================================================

print("CRIAR API GATEWAY")
print("=" * 50)

API_NAME = "lstm-vale3-api"

# Cria HTTP API integrada com Lambda
create_api_cmd = f"""aws apigatewayv2 create-api \
    --name {API_NAME} \
    --protocol-type HTTP \
    --target arn:aws:lambda:{AWS_REGION}:{AWS_ACCOUNT_ID}:function:{LAMBDA_FUNCTION_NAME} \
    --region {AWS_REGION}"""

result = subprocess.run(create_api_cmd.replace('\\\n', ' ').split(), capture_output=True, text=True)

if result.returncode == 0:
    api_info = json.loads(result.stdout)
    API_ID = api_info['ApiId']
    API_ENDPOINT = api_info['ApiEndpoint']
    print("✓ API Gateway criado com sucesso!")
    print(f"   API ID: {API_ID}")
    print(f"   Endpoint: {API_ENDPOINT}")
else:
    print(f"✗ Erro: {result.stderr}")
    API_ID = None
    API_ENDPOINT = None

In [ ]:
# =============================================================================
# ADICIONAR PERMISSÃO PARA API GATEWAY INVOCAR LAMBDA
# =============================================================================

print("\nADICIONAR PERMISSÃO")
print("=" * 50)

permission_cmd = f"""aws lambda add-permission \
    --function-name {LAMBDA_FUNCTION_NAME} \
    --statement-id apigateway-access \
    --action lambda:InvokeFunction \
    --principal apigateway.amazonaws.com \
    --source-arn arn:aws:execute-api:{AWS_REGION}:{AWS_ACCOUNT_ID}:{API_ID}/* \
    --region {AWS_REGION}"""

result = subprocess.run(permission_cmd.replace('\\\n', ' ').split(), capture_output=True, text=True)

if result.returncode == 0:
    print("✓ Permissão adicionada!")
elif "ResourceConflictException" in result.stderr:
    print("✓ Permissão já existe (ok)")
else:
    print(f"✗ Erro: {result.stderr}")

## 10. Testar a API em Produção!

In [ ]:
# =============================================================================
# URL FINAL DA API
# =============================================================================

print("=" * 60)
print("🎉 SUA API ESTÁ NO AR!")
print("=" * 60)

if API_ENDPOINT:
    print(f"\n🔗 URL Base: {API_ENDPOINT}")
    print(f"\n📚 Endpoints disponíveis:")
    print(f"   GET  {API_ENDPOINT}/              → Info da API")
    print(f"   GET  {API_ENDPOINT}/health        → Health check")
    print(f"   GET  {API_ENDPOINT}/modelo/info   → Info do modelo")
    print(f"   GET  {API_ENDPOINT}/predict/latest → Previsão automática")
    print(f"   POST {API_ENDPOINT}/predict       → Previsão manual")
    print(f"   GET  {API_ENDPOINT}/docs          → Documentação Swagger")
else:
    print("\n⚠️ API_ENDPOINT não definido. Verifique os passos anteriores.")

In [ ]:
# =============================================================================
# TESTAR A API
# =============================================================================

import requests

print("TESTANDO A API")
print("=" * 50)

if API_ENDPOINT:
    # Teste 1: Health Check
    print("\n1. Health Check...")
    try:
        response = requests.get(f"{API_ENDPOINT}/health", timeout=60)
        print(f"   Status: {response.status_code}")
        print(f"   Response: {response.json()}")
    except Exception as e:
        print(f"   Erro: {e}")

    # Teste 2: Info do Modelo
    print("\n2. Info do Modelo...")
    try:
        response = requests.get(f"{API_ENDPOINT}/modelo/info", timeout=60)
        print(f"   Status: {response.status_code}")
        if response.status_code == 200:
            info = response.json()
            print(f"   Janela temporal: {info.get('janela_temporal')} dias")
            print(f"   MAPE: {info.get('metricas_teste', {}).get('mape', 'N/A')}%")
    except Exception as e:
        print(f"   Erro: {e}")

    # Teste 3: Previsão
    print("\n3. Previsão com dados atuais...")
    print("   (isso pode demorar ~30 segundos na primeira execução)")
    try:
        response = requests.get(f"{API_ENDPOINT}/predict/latest", timeout=120)
        print(f"   Status: {response.status_code}")
        if response.status_code == 200:
            pred = response.json()
            print(f"\n   📈 PREVISÃO:")
            print(f"   Último preço:    R$ {pred.get('ultimo_preco', 'N/A')}")
            print(f"   Preço previsto:  R$ {pred.get('preco_previsto', 'N/A')}")
            print(f"   Variação:        {pred.get('variacao_percentual', 'N/A')}%")
            print(f"   Direção:         {pred.get('direcao', 'N/A').upper()}")
        else:
            print(f"   Response: {response.text}")
    except Exception as e:
        print(f"   Erro: {e}")
else:
    print("⚠️ API_ENDPOINT não definido.")

In [ ]:
# =============================================================================
# SALVAR INFORMAÇÕES DO DEPLOY
# =============================================================================

deploy_info = {
    'api_endpoint': API_ENDPOINT,
    'api_id': API_ID,
    'lambda_function': LAMBDA_FUNCTION_NAME,
    'lambda_arn': LAMBDA_ARN if 'LAMBDA_ARN' in dir() else None,
    'ecr_uri': ECR_URI,
    'role_name': ROLE_NAME,
    'region': AWS_REGION,
    'account_id': AWS_ACCOUNT_ID
}

with open('../deploy_info.json', 'w') as f:
    json.dump(deploy_info, f, indent=2)

print("\n✓ Informações do deploy salvas em deploy_info.json")
print("\n" + json.dumps(deploy_info, indent=2))

## 11. 🚨 CHECKLIST DE LIMPEZA (APÓS O VÍDEO)

### IMPORTANTE: Execute estes comandos APÓS gravar seu vídeo!

Isso evitará cobranças futuras.

In [ ]:
# =============================================================================
# ⚠️ SCRIPT DE LIMPEZA - EXECUTE APÓS GRAVAR O VÍDEO!
# =============================================================================

print("=" * 60)
print("🧹 SCRIPT DE LIMPEZA - NÃO EXECUTE AGORA!")
print("=" * 60)
print("\nExecute APENAS após gravar seu vídeo de demonstração.")
print("\nDescomente e execute célula abaixo quando estiver pronto.")

In [ ]:
# =============================================================================
# 🗑️ EXECUTAR LIMPEZA (DESCOMENTE QUANDO PRONTO)
# =============================================================================

'''
# DESCOMENTE TUDO ABAIXO PARA EXECUTAR A LIMPEZA

import subprocess
import json

# Carrega informações do deploy
with open('../deploy_info.json', 'r') as f:
    info = json.load(f)

print("INICIANDO LIMPEZA DOS RECURSOS AWS")
print("=" * 50)

# 1. Deletar API Gateway
print("\n1. Deletando API Gateway...")
cmd = f"aws apigatewayv2 delete-api --api-id {info['api_id']} --region {info['region']}"
result = subprocess.run(cmd.split(), capture_output=True, text=True)
print("   ✓ API Gateway deletado" if result.returncode == 0 else f"   ✗ Erro: {result.stderr}")

# 2. Deletar função Lambda
print("\n2. Deletando função Lambda...")
cmd = f"aws lambda delete-function --function-name {info['lambda_function']} --region {info['region']}"
result = subprocess.run(cmd.split(), capture_output=True, text=True)
print("   ✓ Lambda deletado" if result.returncode == 0 else f"   ✗ Erro: {result.stderr}")

# 3. Deletar imagens do ECR
print("\n3. Deletando repositório ECR...")
cmd = f"aws ecr delete-repository --repository-name lstm-vale3-api --force --region {info['region']}"
result = subprocess.run(cmd.split(), capture_output=True, text=True)
print("   ✓ ECR deletado" if result.returncode == 0 else f"   ✗ Erro: {result.stderr}")

# 4. Deletar Role IAM
print("\n4. Deletando Role IAM...")
# Primeiro remove a política anexada
cmd = f"aws iam detach-role-policy --role-name {info['role_name']} --policy-arn arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"
subprocess.run(cmd.split(), capture_output=True, text=True)
# Depois deleta a role
cmd = f"aws iam delete-role --role-name {info['role_name']}"
result = subprocess.run(cmd.split(), capture_output=True, text=True)
print("   ✓ Role deletada" if result.returncode == 0 else f"   ✗ Erro: {result.stderr}")

# 5. Deletar logs do CloudWatch
print("\n5. Deletando logs do CloudWatch...")
cmd = f"aws logs delete-log-group --log-group-name /aws/lambda/{info['lambda_function']} --region {info['region']}"
result = subprocess.run(cmd.split(), capture_output=True, text=True)
print("   ✓ Logs deletados" if result.returncode == 0 else f"   ⚠️ Logs podem não existir ainda")

print("\n" + "=" * 50)
print("✓ LIMPEZA CONCLUÍDA!")
print("=" * 50)
print("\n⚠️ Verifique no console AWS se tudo foi removido.")
print("   Acesse: https://console.aws.amazon.com/billing/")
'''

## 12. Resumo Final

In [ ]:
# =============================================================================
# RESUMO DO DEPLOY
# =============================================================================

print("=" * 60)
print("🎉 DEPLOY CONCLUÍDO COM SUCESSO!")
print("=" * 60)

print(f"\n🔗 URL DA SUA API:")
print(f"   {API_ENDPOINT}")

print(f"\n📚 ENDPOINTS:")
print(f"   • Documentação: {API_ENDPOINT}/docs")
print(f"   • Health:       {API_ENDPOINT}/health")
print(f"   • Previsão:     {API_ENDPOINT}/predict/latest")

print(f"\n💰 CUSTO ESTIMADO:")
print(f"   < $0.50 por mês (com uso mínimo)")

print(f"\n⚠️ LEMBRE-SE:")
print(f"   Após gravar o vídeo, execute o script de limpeza!")
print(f"   (Célula anterior com código comentado)")

print("\n" + "=" * 60)
print("PRÓXIMOS PASSOS")
print("=" * 60)
print("\n1. ✓ Testar a API (feito acima)")
print("2. □ Gravar vídeo de demonstração")
print("3. □ Preparar documentação final")
print("4. □ Executar script de limpeza")
print("5. □ Verificar billing zerado")